In [13]:
import pandas as pd

# Carregar dados do novo path (já balanceado)
df = pd.read_csv("../data/processed/all_datasets_aligned_balanced.csv")
print(df.head())          # Ver as primeiras linhas
print(df['flag'].value_counts())  # Quantos exemplos tem de cada classe
print(f"Dataset shape: {df.shape}")

C:\Users\thelo\AppData\Local\Temp\ipykernel_16692\3542706022.py:4: DtypeWarning: Columns (0: val4, 1: val7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/all_datasets_aligned_balanced.csv")


     id  val1  val2 val3   val4   val5  val6 val7   val8    flag
0   160    93   126  124    9.0    0.0  31.0  2.0    0.0  BENIGN
1  1087     1    69   96  255.0  107.0   0.0  0.0    0.0    gear
2   790    69    41   36  255.0   41.0  36.0  0.0  255.0     RPM
3  1087     1    69   96  255.0  107.0   0.0  0.0    0.0    gear
4  1520     1     0    R    NaN    NaN   NaN  NaN    NaN     NaN
flag
BENIGN    1000000
RPM        709797
DoS        662184
gear       597252
Fuzzy      491846
Name: count, dtype: int64
Dataset shape: (3722480, 10)


In [14]:
import pandas as pd

# Defina o número de amostras desejado para "BENIGN"
n_samples_benign = 1000_000  # ou 1_000_000, como preferir

# Separe a classe majoritária
benign_df = df[df['flag'] == 'BENIGN']

# Separe as demais classes (minoritárias)
minor_classes_df = df[df['flag'] != 'BENIGN']

# Amostragem aleatória da classe BENIGN
benign_sampled = benign_df.sample(n=n_samples_benign, random_state=42)

# Combine novamente o DataFrame balanceado
df_balanced = pd.concat([benign_sampled, minor_classes_df], axis=0)

# Embaralhar o DataFrame final
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Verificar a nova distribuição
print(df_balanced['flag'].value_counts())

df = df_balanced
#df.to_csv("all_datasets_aligned_balanced.csv", index=False)
del df_balanced, benign_df, minor_classes_df, benign_sampled

flag
BENIGN    1000000
RPM        709797
DoS        662184
gear       597252
Fuzzy      491846
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
df = df.dropna()

# Remove a coluna 'id' se existir
if 'id' in df.columns:
    df = df.drop(columns=['id'])
else:
    print("Coluna 'id' não encontrada, continuando...")

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Codificar rótulos em números
le = LabelEncoder()
df['flag'] = le.fit_transform(df['flag'])

X = df.drop(columns=['flag'])  # Features
y = df['flag']                 # Rótulos

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

del df  # Liberar memória

In [17]:
# Verificar tipos de dados e converter colunas problematicas
print("Tipos de dados antes da conversão:")
print(X_train.dtypes)
print()

# Converter colunas que estão como string para float
for col in ['val3', 'val4', 'val6', 'val7']:
    if col in X_train.columns:
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
        print(f"Convertido {col} para float")

# Remover possíveis NaN criados na conversão
X_train = X_train.fillna(X_train.mean())
X_test = X_test.fillna(X_test.mean())

print("\nTipos de dados após conversão:")
print(X_train.dtypes)

Tipos de dados antes da conversão:
val1      int64
val2      int64
val3        str
val4     object
val5    float64
val6        str
val7     object
val8    float64
dtype: object

Convertido val3 para float
Convertido val4 para float
Convertido val6 para float
Convertido val7 para float

Tipos de dados após conversão:
val1      int64
val2      int64
val3      int64
val4    float64
val5    float64
val6    float64
val7    float64
val8    float64
dtype: object


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib
import json
import os

# XGBClassifier configurado para multiclass
xgb = XGBClassifier(random_state=42, objective='multi:softprob', eval_metric='mlogloss')

# -------------------------------------------------------------------------
# MUDANÇA: Grade de hiperparâmetros específica para o XGBoost
# -------------------------------------------------------------------------
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Validação cruzada estratificada (sem alterações aqui)
# OBS: n_splits=2 é baixo, considere usar 5 ou 10 para uma validação mais robusta.
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# -------------------------------------------------------------------------
# MUDANÇA: Passar o estimador 'xgb' para o GridSearchCV
# -------------------------------------------------------------------------
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, 
                           scoring='accuracy', cv=cv, n_jobs=-1, verbose=3) # n_jobs=-1 usa todos os cores

# Executa a busca
grid_search.fit(X_train, y_train)

# Resultados
print("Melhores parâmetros encontrados:")
print(grid_search.best_params_)
print(f"Melhor acurácia: {grid_search.best_score_:.4f}")

# Para ver o desempenho do melhor modelo no dataset de teste
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Acurácia no test set: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

# Salvar resultados em disco
results_dir = os.path.abspath(os.path.join('..', 'results'))
metrics_dir = os.path.join(results_dir, 'metrics')
os.makedirs(metrics_dir, exist_ok=True)

report = classification_report(y_test, y_pred)
with open(os.path.join(metrics_dir, 'xgboost_classification_report.txt'), 'w', encoding='utf-8') as f:
    f.write(report)

summary = {
    'best_params': grid_search.best_params_,
    'best_cv_accuracy': float(grid_search.best_score_),
    'test_accuracy': float(accuracy),
    'n_test_samples': int(y_test.shape[0])
}
with open(os.path.join(metrics_dir, 'xgboost_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(cm).to_csv(os.path.join(metrics_dir, 'xgboost_confusion_matrix.csv'), index=False)

pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred}).to_csv(
    os.path.join(metrics_dir, 'xgboost_test_predictions.csv'), index=False
)

joblib.dump(best_model, os.path.join(results_dir, 'xgboost_best_model.joblib'))

print(f"Resultados salvos em: {metrics_dir}")
print(f"Modelo salvo em: {os.path.join(results_dir, 'xgboost_best_model.joblib')}")

Fitting 2 folds for each of 48 candidates, totalling 96 fits


c:\Users\thelo\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:07:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Melhores parâmetros encontrados:
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}
Melhor acurácia: 0.9890
              precision    recall  f1-score   support

           0       1.00      0.96      0.98    200000
           1       0.95      1.00      0.97    132437
           2       1.00      1.00      1.00     98369
           3       1.00      1.00      1.00    141960
           4       1.00      1.00      1.00    119450

    accuracy                           0.99    692216
   macro avg       0.99      0.99      0.99    692216
weighted avg       0.99      0.99      0.99    692216



In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Código SVM com normalização e GridSearch
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(probability=True, random_state=42))
])

param_grid_svm = {
    'svc__kernel': ['rbf', 'linear'],
    'svc__C': [0.1, 1, 10],
    'svc__gamma': ['scale', 'auto']
}

svm_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid_svm,
    scoring='accuracy',
    cv=svm_cv,
    n_jobs=-1,
    verbose=3
)

svm_search.fit(X_train, y_train)

print('Melhores parâmetros SVM:')
print(svm_search.best_params_)
print(f"Melhor acurácia SVM (CV): {svm_search.best_score_:.4f}")

svm_best = svm_search.best_estimator_
svm_pred = svm_best.predict(X_test)
print('Relatório de classificação SVM:')
print(classification_report(y_test, svm_pred))

# Opcional: salvar resultados SVM
dirs = os.path.abspath(os.path.join('..', 'results'))
os.makedirs(os.path.join(dirs, 'metrics'), exist_ok=True)

svm_summary = {
    'best_params': svm_search.best_params_,
    'best_cv_accuracy': float(svm_search.best_score_),
    'test_accuracy': float(accuracy_score(y_test, svm_pred)),
    'n_test_samples': int(y_test.shape[0])
}
with open(os.path.join(dirs, 'metrics', 'svm_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(svm_summary, f, indent=2)

pd.DataFrame({'y_true': y_test.values, 'y_pred': svm_pred}).to_csv(
    os.path.join(dirs, 'metrics', 'svm_test_predictions.csv'), index=False
)

joblib.dump(svm_best, os.path.join(dirs, 'svm_best_model.joblib'))
print(f"SVM: resultados salvos em {os.path.join(dirs, 'metrics')}")
print(f"SVM: modelo salvo em {os.path.join(dirs, 'svm_best_model.joblib')}")